# monogate Neurosymbolic Prover — Showcase

> **v1.1.0** · [GitHub](https://github.com/agent-maestro/monogate) · [arXiv:2603.21852](https://arxiv.org/abs/2603.21852)

monogate started as a study of one operator: `eml(x, y) = exp(x) − ln(y)`.  
Every elementary function is a finite tree built from this single gate.

In v1.1, that same substrate powers a **neurosymbolic theorem prover** that:
- **Proves** mathematical identities across four tiers (numerical → exact → interval → witness)
- **Generates conjectures** from the EML expression space
- **Compresses** proof trees to minimal form
- **Visualizes** proofs interactively with Plotly
- **Learns** from every successful proof via an online neural scorer

This notebook walks through all of these capabilities.

In [ ]:
# Install (uncomment if running in Colab)
# !pip install -q "monogate[sympy]" plotly

In [ ]:
import monogate
print(f"monogate {monogate.__version__}")

---
## 1 — The Four-Tier Proof Pipeline

Every call to `prove()` runs up to four tiers in order, stopping as soon as one succeeds:

| Tier | Method | Output |
|------|--------|--------|
| 1 | Numerical probe (500 random points) | Refutation if any point fails |
| 2 | SymPy exact simplification | `proved_exact` with confidence 1.0 |
| 3 | Interval arithmetic certification | `proved_certified` |
| 4 | EML witness search (MCTS) | `proved_witness` + expression tree |

In [ ]:
from monogate.prover import EMLProverV2

p = EMLProverV2(enable_learning=True)   # neural scorer active

# --- Tier 2: SymPy exact ---
r = p.prove("sin(x)**2 + cos(x)**2 == 1")
print(f"sin²+cos²==1 → {r.status}  confidence={r.confidence}")

r = p.prove("exp(x + y) == exp(x) * exp(y)")
print(f"exp addition  → {r.status}  confidence={r.confidence}")

r = p.prove("cosh(x)**2 - sinh(x)**2 == 1")
print(f"hyperbolic Pythagorean → {r.status}  confidence={r.confidence}")

# --- Tier 1 refutation: false identity ---
r_false = p.prove("sin(x) == cos(x)")
print(f"sin==cos (false) → {r_false.status}")

---
## 2 — Batch Proving: How Well Does It Do on the Full Catalog?

In [ ]:
from monogate.identities import (
    TRIG_IDENTITIES, HYPERBOLIC_IDENTITIES, EXPONENTIAL_IDENTITIES,
    ALL_IDENTITIES,
)

print(f"Total catalog: {len(ALL_IDENTITIES)} identities")

# Prove all trig identities
p2 = EMLProverV2()
report = p2.batch_prove(TRIG_IDENTITIES)

print(f"\nTrigonometric identities ({len(TRIG_IDENTITIES)} total):")
print(f"  Proved:  {report.n_proved}")
print(f"  Failed:  {report.n_failed}")
print(f"  Rate:    {report.proved_rate:.0%}")
print(f"  Time:    {report.total_time:.1f}s")

In [ ]:
# Show the per-identity results
for entry in report.results[:10]:     # first 10
    icon = "✓" if entry.result.proved() else "✗"
    print(f"  {icon} {entry.identity.name:35s}  {entry.result.status}")

---
## 3 — Physics Identities: PDEs and Solitons

In [ ]:
from monogate.identities import get_by_category

physics = get_by_category("physics")
print(f"Physics identities: {len(physics)}\n")

p3 = EMLProverV2()
for identity in physics[:6]:
    r = p3.prove(identity.expression)
    icon = "✓" if r.proved() else "✗"
    print(f"  {icon} {identity.name:40s}  {r.status}")

---
## 4 — Conjecture Generation: Mathematical Discovery

In [ ]:
p4 = EMLProverV2()

print("Generating conjectures from EML expression space...\n")
candidates = p4.generate_conjectures(n=30)

print(f"Found {len(candidates)} candidate identities:\n")
for c in candidates[:10]:
    print(f"  conf={c.confidence:.2f}  {c.expression}")

In [ ]:
# Try to prove the top conjecture
if candidates:
    top = candidates[0]
    print(f"Attempting to prove: {top.expression}\n")
    r = p4.prove(top.expression)
    print(f"Result: {r.status}  confidence={r.confidence}")
    if r.proved() and r.node_count:
        print(f"Witness tree: {r.node_count} nodes")

---
## 5 — Proof Compression

Witness trees from MCTS tier 4 can be compressed via minimax approximation.

In [ ]:
p5 = EMLProverV2()

# Prove an identity that uses the witness tier
r = p5.prove("exp(2*x) == exp(x)**2")
print(f"Status: {r.status}")

if r.proved() and r.node_count:
    print(f"Before compression: {r.node_count} nodes")
    compressed = p5.compress_proof(r)
    print(f"After compression:  {compressed.node_count} nodes")
    reduction = 1 - compressed.node_count / r.node_count
    print(f"Reduction: {reduction:.0%}")

---
## 6 — Interactive Proof Visualization

The prover can render any proof tree as an interactive Plotly figure.  
Node colors: **blue** = EML operator · **green** = constant · **red** = variable x

In [ ]:
try:
    import plotly  # noqa: F401
    HAS_PLOTLY = True
except ImportError:
    HAS_PLOTLY = False
    print("plotly not installed. Run: pip install plotly")

if HAS_PLOTLY:
    p6 = EMLProverV2()
    result = p6.prove("sin(x)**2 + cos(x)**2 == 1")
    fig = p6.visualize_proof_interactive(result)
    fig.show()

In [ ]:
# Export to HTML (shareable, no server required)
if HAS_PLOTLY:
    p6b = EMLProverV2()
    result2 = p6b.prove("cosh(x)**2 - sinh(x)**2 == 1")
    p6b.visualize_proof_interactive(result2, output_path="hyperbolic_proof.html")
    print("Saved: hyperbolic_proof.html")

---
## 7 — Online Neural Scorer

The `FeatureBasedEMLScorer` learns from proved identities and guides MCTS
toward more promising expression trees. It requires no torch for inference —
weights are stored as plain numpy arrays.

In [ ]:
from monogate import FeatureBasedEMLScorer, extract_tree_features, N_FEATURES
from monogate.search.mcts import _eml, _leaf
import numpy as np

print(f"Feature vector dimension: {N_FEATURES}")

# Build an example tree: eml(x, 1.0)
tree = _eml(_leaf("x"), _leaf(1.0))
feat = extract_tree_features(tree)

feature_names = [
    "depth", "eml_nodes", "total_nodes", "leaf_ratio",
    "x_fraction", "const_mean", "const_std", "const_range",
    "balance", "symmetry", "max_const", "depth_variance",
]
print("\nFeatures for eml(x, 1.0):")
for name, val in zip(feature_names, feat):
    print(f"  {name:16s} {val:.4f}")

In [ ]:
# Lifecycle: untrained → trained → save/load
scorer = FeatureBasedEMLScorer(min_samples=5, retrain_every=5)

print(f"Untrained score: {scorer.score(tree):.3f}  (should be 0.5)")
print(f"is_trained(): {scorer.is_trained()}")

# Simulate receiving rewards from 10 proved witnesses
for i in range(10):
    reward = 1.0 / (1.0 + (i % 5))   # smaller trees → higher reward
    scorer.update(tree, reward)

print(f"\nAfter 10 updates:")
print(f"is_trained(): {scorer.is_trained()}")
print(f"Score:        {scorer.score(tree):.4f}")

In [ ]:
# Save and reload
scorer.save("/tmp/scorer.json")

scorer2 = FeatureBasedEMLScorer()
scorer2.load("/tmp/scorer.json")

import math
diff = abs(scorer.score(tree) - scorer2.score(tree))
print(f"Round-trip score difference: {diff:.2e}  (should be ~0)")

---
## 8 — Learning Progression

With `enable_learning=True`, the prover updates the scorer after every
witness-tier proof. As the scorer accumulates experience, it can guide
MCTS more effectively.

In [ ]:
from monogate.identities import get_by_difficulty

# Use easy identities to warm up the scorer
easy = get_by_difficulty("easy")
print(f"Easy identities: {len(easy)}")

p8 = EMLProverV2(enable_learning=True)

proved = 0
for identity in easy[:15]:
    r = p8.prove(identity.expression)
    if r.proved():
        proved += 1

print(f"Proved {proved}/{min(15, len(easy))} easy identities")
print(f"Scorer buffer size: {len(p8.scorer._buffer)}")
print(f"Scorer is_trained: {p8.scorer.is_trained()}")

---
## 9 — Wow Examples: Surprising Identities

Some of the more striking entries in the catalog.

In [ ]:
wow_identities = [
    # EML constructions
    "exp(ln(x)) == x",
    "ln(exp(x) - 1 + 1) == x",
    # Chebyshev
    "16*sin(x)**5 - 20*sin(x)**3 + 5*sin(x) == sin(5*x)",
    # Soliton amplitude
    "2 / cosh(x) ** 2 == 2 - 2*tanh(x)**2",
]

p9 = EMLProverV2()
print("Identity verification:\n")
for expr in wow_identities:
    r = p9.prove(expr)
    icon = "✓" if r.proved() else "✗"
    tier = r.status.replace("proved_", "").replace("disproved", "DISPROVED")
    print(f"  {icon} [{tier:12s}]  {expr}")

---
## 10 — Full Quick Reference

```python
from monogate.prover import EMLProverV2
from monogate.identities import ALL_IDENTITIES, get_by_category, get_by_difficulty
from monogate import FeatureBasedEMLScorer, extract_tree_features, N_FEATURES

# Create prover
p = EMLProverV2(enable_learning=True)

# Prove a single identity
r = p.prove("sin(x)**2 + cos(x)**2 == 1")
r.proved()          # True
r.status            # 'proved_exact'
r.confidence        # 1.0
r.node_count        # tree size (witness tier only)

# Batch proving
report = p.batch_prove(ALL_IDENTITIES)
report.proved_rate  # fraction proved
report.results      # list of (identity, ProofResult)

# Conjecture generation
candidates = p.generate_conjectures(n=20, category="trig")

# Proof compression
compressed = p.compress_proof(r)

# Interactive visualization
fig = p.visualize_proof_interactive(r, output_path="proof.html")

# Persist learned scorer
p.scorer.save("scorer.json")
```

**Links:** [GitHub](https://github.com/agent-maestro/monogate) · [PyPI](https://pypi.org/project/monogate/) · [arXiv](https://arxiv.org/abs/2603.21852) · [monogate.dev](https://monogate.dev)